# BTS Digital Twin (NVS) — Model 2: mạng sửa lỗi pixel (image-domain corrector)

**Nhánh code:** `feature/nn-image-corrector` (KHÔNG nằm trên `main` — kiến trúc thử
nghiệm, tách biệt hoàn toàn với cơ chế "error-guided refine" Vòng 2/3 vốn train tiếp
CÙNG 1 bộ Gaussian 3D). Xem `docs/MILESTONE_15_image_corrector.md` để biết đầy đủ thiết
kế + rủi ro.

Notebook này **KHÔNG train Model 1** (3D Gaussian Splatting) — dùng lại nguyên vẹn
`kaggle_round1_baseline.ipynb` (không sửa) để train Model 1, **khuyến nghị chạy với
`MODE="final"` (100% ảnh train, 30000 iteration)** rồi tải `gs_model/` lên Drive như
bình thường.

Notebook này train **Model 2** — 1 mạng neural 2D HOÀN TOÀN RIÊNG BIỆT
(`pipeline/common/corrector_model.py::ResidualCorrectorNet`), nhận ảnh Model 1 render
tại pose TRAIN làm input, học sửa/làm nét dựa trên so ảnh đó với GT thật. Sau khi train
xong, áp lên ảnh Model 1 render tại pose TEST thật (`test_poses.csv`) rồi mới đóng gói
nộp bài.

**KHÔNG có holdout / Score khách quan cho nhánh này** (quyết định có chủ đích) — chỉ có
thể kiểm tra chất lượng bằng mắt (Bước 12 bên dưới). Xem mục rủi ro trong milestone doc
trước khi dùng ảnh đã sửa để nộp bài thật.


## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
# DỪNG NGAY nếu không có GPU — xem lý do y hệt ở kaggle_round1_baseline.ipynb Bước 1
# (khỏi lặp lại toàn bộ comment ở đây, tinh thần giống hệt: crash muộn/mù mờ trên CPU
# còn tốn thời gian hơn nhiều so với dừng sớm ở đây).
if not torch.cuda.is_available():
    raise SystemExit(
        "KHÔNG có GPU khả dụng (torch.cuda.is_available()=False). Vào Settings (góc phải) "
        "-> Accelerator -> chọn GPU T4 x2 hoặc P100 -> Save, rồi chạy lại từ đầu. "
        "Bước 8 (sinh dataset) cần CUDA để render Model 1 — chạy trên CPU sẽ crash muộn/treo."
    )
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip install -q "pycolmap>=3.10" "scikit-image>=0.19" lpips plyfile tqdm opencv-python-headless "gdown>=6,<7"
# Không cần cài thêm gì cho Model 2 (torch/torchvision đã có sẵn trong image Kaggle,
# corrector_model.py chỉ dùng torch.nn thuần, không có dependency nào mới.)

## Bước 2 — Clone + build 3D Gaussian Splatting

Vẫn cần GS_REPO cho **Bước 8** (`08_build_corrector_dataset.py` render lại pose TRAIN
bằng Model 1) — giống hệt lý do các notebook Vòng 1/2/3 cần. Pin đúng commit đã xác
nhận có `--antialiasing` (mip-splatting tích hợp sẵn), khớp checkpoint Model 1 đã train.


In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])


## Bước 3 — Lấy code pipeline từ Git repo của bạn

**`GIT_BRANCH` mặc định trỏ sang `feature/nn-image-corrector`** (khác các notebook
Vòng 1/2/3/submission vốn trỏ `main`) — vì `08_build_corrector_dataset.py`/
`09_train_corrector.py`/`10_apply_corrector.py`/`pipeline/common/corrector_model.py`
CHỈ tồn tại trên nhánh này. Đổi lại `"main"` (hoặc nhánh khác) nếu bạn đã merge nhánh
này vào `main`.


In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin-MultiRound.git"
GIT_BRANCH = "feature/nn-image-corrector"  # <-- đổi nếu đã merge vào main hoặc dùng nhánh khác

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone


In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
# Y hệt logic ở kaggle_round1_baseline.ipynb/kaggle_round2_refine.ipynb Bước 3.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại GIT_BRANCH có đúng "
        "'feature/nn-image-corrector' (hoặc nhánh đã merge nhánh này) không."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

# Kiểm tra sớm: script Model 2 phải có mặt (chặn nhầm nhánh main không có các file này).
for _rel in ("scripts/08_build_corrector_dataset.py", "scripts/09_train_corrector.py",
             "scripts/10_apply_corrector.py", "common/corrector_model.py"):
    _p = target / _rel
    if not _p.exists():
        raise SystemExit(
            f"Thiếu {_p} — GIT_BRANCH='{GIT_BRANCH}' có vẻ CHƯA có code Model 2 "
            "(kiểm tra lại đã trỏ đúng nhánh feature/nn-image-corrector hoặc nhánh đã merge nó chưa)."
        )
print("Đủ file Model 2 (08/09/10 + corrector_model.py).")


## Bước 4 — Tải dataset từ Google Drive

Giống hệt các notebook Vòng 1/2/3 — điền link chia sẻ Google Drive (chế độ "Anyone
with the link") vào `GDRIVE_URL`.


In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục chứa các scene ...")


In [ ]:
# Tự dò thư mục chứa các scene phẳng + symlink về vị trí common/scenes.py cần.
# Y hệt logic ở kaggle_round1_baseline.ipynb/kaggle_round2_refine.ipynb Bước 4.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        f"Không tìm thấy thư mục nào chứa >= {_MIN_MATCH}/{len(_expected_scene_dirs)} scene mong đợi "
        f"bên trong {RAW_ROOT}. Khớp tốt nhất: {found} ({best_match} scene)."
    )

print(f"Tìm thấy thư mục dataset tại: {found} ({best_match}/{len(_expected_scene_dirs)} scene khớp)")

target_parent = Path("/kaggle/working/Dataset")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "VAI_NVS_DATA_ROUND2"
if target.is_symlink() or target.exists():
    if target.is_symlink():
        target.unlink()
    else:
        import shutil
        shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

os.environ["BTS_DATASET_ROOT"] = str(target)
print("BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")


## Bước 5 — Cấu hình scene + checkpoint Model 1 (khuyến nghị train 100% dữ liệu)

- `SCENE`: 1 trong 7 tên scene.
- `CHECKPOINT_DRIVE_LINK`: link Drive tới thư mục `gs_model/` của Model 1 — **khuyến
  nghị dùng checkpoint train ở `MODE="final"` (100% ảnh train, 30000 iteration) của
  `kaggle_round1_baseline.ipynb`**, đúng theo ý tưởng của nhánh này (không cần giữ lại
  holdout cho Model 1 — khác hẳn nhánh `main`/Vòng 2-3, xem milestone doc). Script
  KHÔNG ép buộc `train_mode`, chỉ khuyến nghị.


In [ ]:
SCENE = "HCM0421"  # <-- đổi thành tên scene muốn xử lý
CHECKPOINT_DRIVE_LINK = ""  # <-- dán link Drive tới thư mục gs_model của Model 1 (khuyến nghị MODE="final")
STEPS = 4000        # <-- số bước train Model 2 LẦN ĐẦU (xem Bước 9)

assert CHECKPOINT_DRIVE_LINK, "Chưa điền CHECKPOINT_DRIVE_LINK."
print(f"SCENE={SCENE}  STEPS={STEPS}")


## Bước 6 — Tải checkpoint Model 1 từ Google Drive

In [ ]:
import shutil
from pathlib import Path

dest_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}")
dest_dir.mkdir(parents=True, exist_ok=True)
gs_model_dst = dest_dir / "gs_model"
shutil.rmtree(gs_model_dst, ignore_errors=True)

raw_dl_dir = Path(f"/kaggle/working/_ckpt_raw/{SCENE}")
shutil.rmtree(raw_dl_dir, ignore_errors=True)
raw_dl_dir.mkdir(parents=True, exist_ok=True)
print(f"===== {SCENE}: tải thư mục gs_model từ Drive =====")
!gdown --folder "{CHECKPOINT_DRIVE_LINK}" -O "{raw_dl_dir}"

candidates = [p.parent for p in raw_dl_dir.rglob("cfg_args")]
assert candidates, (
    f"Tải xong nhưng KHÔNG tìm thấy file 'cfg_args' trong {raw_dl_dir} — kiểm tra lại "
    f"link Drive có đúng là THƯ MỤC gs_model/ (không phải chỉ mỗi point_cloud.ply) và đã "
    f"share \"Anyone with the link\" chưa.")
src_root = candidates[0]
assert (src_root / "pipeline_train_flags.json").exists(), (
    f"Thiếu pipeline_train_flags.json trong {src_root} — thư mục gs_model tải lên Drive "
    f"phải nguyên vẹn (không tự xoá bớt file con nào).")

# Cross-check SCENE (biến notebook) với scene THẬT của checkpoint vừa tải — cùng cơ chế
# đã thêm ở kaggle_round2_refine.ipynb/kaggle_round3_refine.ipynb (pass verification #11,
# xem docs/PORTED_KNOWLEDGE.md mục 6g và MILESTONE_14) để tránh dán NHẦM link Drive của
# 1 scene KHÁC — ở đây áp dụng lại nguyên vẹn logic đó.
from argparse import Namespace as _NS
_cfg_scene_guess = None
_cfg_source_path = None
try:
    _cfg_ns = eval((src_root / "cfg_args").read_text(), {"Namespace": _NS})
    _cfg_source_path = getattr(_cfg_ns, "source_path", None)
    if _cfg_source_path:
        _cfg_scene_guess = Path(_cfg_source_path).parent.parent.name
except Exception as e:
    print(f"[CẢNH BÁO] Không đọc được source_path từ {src_root / 'cfg_args'}: {e} — bỏ qua cross-check scene.")
if _cfg_scene_guess and _cfg_scene_guess != SCENE:
    raise SystemExit(
        f"{SCENE}: checkpoint vừa tải về có vẻ THUỘC SCENE KHÁC ('{_cfg_scene_guess}', suy ra từ "
        f"cfg_args.source_path='{_cfg_source_path}') — kiểm tra lại CHECKPOINT_DRIVE_LINK."
    )

import json as _json
_flags = _json.loads((src_root / "pipeline_train_flags.json").read_text())
_train_mode = _flags.get("train_mode")
if _train_mode != "final":
    print(f"  [LƯU Ý] pipeline_train_flags.json ghi train_mode='{_train_mode}' (không phải 'final') "
          "— nhánh này KHUYẾN NGHỊ dùng checkpoint train trên 100% dữ liệu ('final'), nhưng KHÔNG "
          "chặn cứng nếu bạn cố ý dùng checkpoint khác (vd đang thử nghiệm với checkpoint holdout).")

shutil.copytree(src_root, gs_model_dst)
shutil.rmtree(raw_dl_dir, ignore_errors=True)

CKPT_ITERATION = max(int(p.name.split("_")[-1]) for p in (gs_model_dst / "point_cloud").glob("iteration_*"))
print(f"-> OK, {gs_model_dst} — checkpoint Model 1 ở iteration {CKPT_ITERATION}")
print(f"-> pipeline_train_flags.json: {_flags}")


## Bước 7 — Tái tạo `colmap/dense/{images/,sparse/0/}` (100% ảnh, KHÔNG `--holdout`)

Bước train Model 1 trước đó đã tự dọn `dense/images/` sau khi xong (dọn đĩa bình
thường, xem `02_train_baseline.sh`). `08_build_corrector_dataset.py` (Bước 8) cần ảnh
ĐÃ undistort chính xác pixel-for-pixel để so với render — phải tái tạo lại TRƯỚC.
**KHÔNG dùng `--holdout`** (khác `kaggle_round2_refine.ipynb`) — nhánh này train Model 1
trên 100% dữ liệu, không giữ lại phần holdout nào.


In [ ]:
!python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE}

## Bước 8 — Sinh dataset cặp (render, GT) cho Model 2

Render lại TOÀN BỘ pose ảnh train bằng checkpoint Model 1 vừa tải, lưu cặp (render,
GT thật) — chỉ cần chạy 1 lần cho mỗi checkpoint Model 1 (Bước 9/10 sau đó chỉ cần
`torch` thuần, không cần GS_REPO/CUDA nữa).


In [ ]:
!python /kaggle/working/pipeline/scripts/08_build_corrector_dataset.py --scene {SCENE}

## Bước 9 — Train Model 2 (lần đầu)

`STEPS` đặt ở Bước 5 (mặc định 4000) — train từ đầu (chưa có checkpoint Model 2 nào).


In [ ]:
!python /kaggle/working/pipeline/scripts/09_train_corrector.py --scene {SCENE} --steps {STEPS}

## Bước 10 (tuỳ chọn) — Train tiếp "lần 2" / "lần 3"...

Đúng ý tưởng ban đầu: **muốn tinh chỉnh thêm, chỉ cần chạy LẠI cell dưới đây** (đổi
`MORE_STEPS` nếu muốn) — `--resume` nạp checkpoint Model 2 hiện có, train tiếp thêm
đúng số bước chỉ định, KHÔNG train lại từ đầu. Chạy cell này bao nhiêu lần tuỳ ý.


In [ ]:
MORE_STEPS = 2000  # <-- số bước train THÊM mỗi lần chạy lại cell này

!python /kaggle/working/pipeline/scripts/09_train_corrector.py --scene {SCENE} --resume --steps {MORE_STEPS}


## Bước 11 — Render `test_poses.csv` thật bằng Model 1, rồi áp Model 2

`03_render_test_poses.py` **KHÔNG bị sửa** ở nhánh này — dùng lại y nguyên, render bằng
Model 1 vào `pipeline/work/<scene>/renders/` như bình thường. `10_apply_corrector.py`
đọc đúng thư mục đó, áp Model 2 (tiled inference), ghi kết quả sang
`pipeline/work_corrected/<scene>/renders/`.


In [ ]:
!python /kaggle/working/pipeline/scripts/03_render_test_poses.py --scene {SCENE}
!python /kaggle/working/pipeline/scripts/10_apply_corrector.py --scene {SCENE}


### Bước 12 — Xem thử vài ảnh TRƯỚC/SAU (kiểm tra bằng mắt — thay cho Score khách quan)

**Nhánh này KHÔNG có Score định lượng** (không holdout, xem milestone doc mục rủi ro) —
đây là bước kiểm tra DUY NHẤT trước khi quyết định có dùng ảnh đã sửa để nộp bài hay
không. Nếu ảnh "sau" trông TỆ hơn "trước" (nhiễu/artifact lạ) cho scene này, đừng dùng
`work_corrected/` cho scene đó — dùng thẳng `pipeline/work/<scene>/renders/` (Model 1
gốc, chưa sửa) khi đóng gói (xem Bước cuối).


In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

before_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}/renders")
after_dir = Path(f"/kaggle/working/pipeline/work_corrected/{SCENE}/renders")
sample_names = sorted(p.name for p in before_dir.glob("*.png"))[:3]

for name in sample_names:
    print(f"--- {name} (trái: Model 1 gốc | phải: sau Model 2) ---")
    im_before = Image.open(before_dir / name)
    im_after = Image.open(after_dir / name)
    w, h = im_before.size
    combo = Image.new("RGB", (w * 2 + 10, h), (255, 255, 255))
    combo.paste(im_before, (0, 0))
    combo.paste(im_after, (w + 10, 0))
    display(combo.resize((min(1200, combo.width), int(combo.height * min(1200, combo.width) / combo.width))))


## Bước 13 — Lưu checkpoint Model 2 lên Google Drive

Giống Bước 6/12 của `kaggle_round1_baseline.ipynb`/`kaggle_round2_refine.ipynb`: bấm
**Save Version**, vào tab **Output**, tìm `pipeline/work/<SCENE>/corrector_model/`, tải
về rồi upload nguyên thư mục lên Drive (đặt tên rõ theo scene, vd
`<SCENE>_corrector_model`) — cần lại nếu muốn train tiếp "lần N" ở phiên Kaggle khác
(dùng `--resume`, xem Bước 10), không cần nếu chỉ chạy 1 phiên duy nhất tới khi đóng
gói xong.


In [ ]:
print(f"Checkpoint Model 2: pipeline/work/{SCENE}/corrector_model/corrector.pt")
print(f"Render đã sửa: pipeline/work_corrected/{SCENE}/renders/")


## Đóng gói submission

`07_package_submission.py` **KHÔNG bị sửa** ở nhánh này — script đó vốn đã có sẵn
`--renders_root` để trỏ sang thư mục render khác thay vì `pipeline/work` mặc định.
Lặp lại toàn bộ notebook này (Bước 5-13) cho từng scene muốn áp Model 2, rồi đóng gói:

```
python pipeline/scripts/07_package_submission.py \
    --renders_root pipeline/work_corrected \
    --out submission.zip
```

Scene NÀO CHƯA chạy qua notebook này (chưa có `work_corrected/<scene>/renders/`) sẽ
KHÔNG có trong `pipeline/work_corrected/` — với những scene đó, hoặc (a) chạy
`10_apply_corrector.py --scene <scene>` (sẽ tự copy-through render Model 1 gốc nếu chưa
có corrector, xem docstring script đó), hoặc (b) đóng gói riêng scene đó từ
`pipeline/work` mặc định (không dùng Model 2) rồi gộp file zip thủ công.
